# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row represents the daily performance of one specific piece of content (`content_hash_id`) per day (`report_date`).

**Time Window:** The mid-panel month of March 2026 (`2026-03`). I am deliberately targeting this specific partitioned month to avoid the final test month and prevent data leakage from future outcomes.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**1. Context:** 
`client_hash_id`, `content_hash_id`, `report_date` (These identify the row and tie it to a specific client and time, but are not used as mathematical features).

**2. Feature:** 
`gsc_impressions`, `gsc_clicks` (These are knowable at the decision moment and represent the historical performance of the content).

**3. Label (Proxy):** 
`needs_refresh` (A binary proxy label indicating if `gsc_clicks` dropped significantly over a specific historical window).

**4. Excluded:** 
Future traffic metrics (e.g., clicks in the next 30 days). Excluded because we cannot know future traffic at the moment of making a decision; including it would cause data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
import duckdb
import getpass
import os

# 1. التأكد من وجود التوكن
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass("Enter your Hugging Face Read Token: ")

# 2. تعريف مفتاح Hugging Face
duckdb.sql(f"CREATE SECRET IF NOT EXISTS hf_secret (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

# 3. المسار الدقيق لملف شهر مارس 2026
table_path = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'"

# A. Row Counts & Missing Values
query_counts = f"""
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN content_hash_id IS NULL THEN 1 ELSE 0 END) as missing_content_ids
FROM {table_path};
"""
print("1. Counts & Missing Values:")
display(duckdb.sql(query_counts).df())

# B. Grain Check (One row = one content piece per day)
# If the grain is strictly unique, this should return an empty dataframe
query_grain = f"""
SELECT report_date, content_hash_id, COUNT(*) as duplicates
FROM {table_path}
GROUP BY report_date, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""
print("\n2. Grain Check (Should be empty):")
display(duckdb.sql(query_grain).df())

# C. Window Check (Data Sample)
query_window = f"""
SELECT report_date, content_hash_id, gsc_clicks, gsc_impressions
FROM {table_path}
LIMIT 3;
"""
print("\n3. Window Check (Data Sample):")
display(duckdb.sql(query_window).df())

1. Counts & Missing Values:


,total_rows,missing_content_ids
0,9841378,0.0



2. Grain Check (Should be empty):


,report_date,content_hash_id,duplicates



3. Window Check (Data Sample):


,report_date,content_hash_id,gsc_clicks,gsc_impressions
0,2026-03-01,content_b7e512995f79d5a6,0,20
1,2026-03-01,content_05597932fe4da067,0,1
2,2026-03-01,content_7a105f548d9c6916,1,125


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits:**
This dataset relies exclusively on quantitative Google Search Console metrics (like `gsc_clicks` and `gsc_impressions`). It can never tell us the *qualitative* reason why a page is losing traffic. It cannot tell us if a competitor published a better article, or if the facts in our content became obsolete. Additionally, we only know if data is available (`gsc_data_available`), but not the full spectrum of user behavior on the page itself.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.